In [10]:
# import libraries
import os
from azure.ai.ml import MLClient, command, Input, Output
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes, InputOutputModes

subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID")
resource_group = os.environ.get("AZURE_RESOURCE_GROUP")
workspace_name = "myworkspace"

# connect to azureml workspace
mlClient = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [ ]:
# create data asset of type uri_file
data = Data(
    name="diabetes-dataasset",
    version="1",
    type=AssetTypes.URI_FILE,
    path="azureml://datastores/myblobdatastore/paths/diabetes.csv"
)

mlClient.data.create_or_update(data)

Data({'path': 'azureml://subscriptions/c94d6fba-0d29-48cd-9cbb-1f115e688235/resourcegroups/ai-300/workspaces/myworkspace/datastores/myblobdatastore/paths/diabetes.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-dataasset', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/c94d6fba-0d29-48cd-9cbb-1f115e688235/resourceGroups/ai-300/providers/Microsoft.MachineLearningServices/workspaces/myworkspace/data/diabetes-dataasset/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\omlln\\Desktop\\My Learning\\ML\\mlops-notebook-classification-experiment', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000001B036124880>, 'serialize': <msrest.serialization.Serializer object at 0x000001B03617A3B0>, 'version': '1', 'latest_version': None, 'datastore': 

In [24]:
#setup and submit job

# inputs
inputs = {
    "dataset": Input(
        path="azureml:diabetes-dataasset:1",
        type=AssetTypes.URI_FILE,
        mode=InputOutputModes.RO_MOUNT
    )
}

# job config
import random

job = command(
    name=f"diabetes-training-{random.randint(1,50)}",
    code="./src",
    command="python training_script.py --input-data ${{inputs.dataset}}",
    inputs=inputs,
    compute="mytargetcomputeinstance",
    environment="my-env:1",
    experiment_name="my-experiments",
    identity=None
)

result = mlClient.jobs.create_or_update(job).studio_url
print(f"Monitor job at: {result}")

Monitor job at: https://ml.azure.com/runs/diabetes-training-16?wsid=/subscriptions/c94d6fba-0d29-48cd-9cbb-1f115e688235/resourcegroups/ai-300/workspaces/myworkspace&tid=d10442b4-3405-4843-9c07-1a34aa407014
